In [3]:
pip install pymysql

  Using cached pymysql-1.1.2-py3-none-any.whl.metadata (4.3 kB)
Using cached pymysql-1.1.2-py3-none-any.whl (45 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import time
import random
import re

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait

from sqlalchemy import create_engine, Column, Integer, Text
from sqlalchemy.orm import declarative_base, sessionmaker


# ==============================
# CONFIG
# ==============================

BASE_URL = "https://www.indiabix.com/data-interpretation/questions-and-answers/"

MIN_DELAY = 2
MAX_DELAY = 4

SECTION = "Data Interpretation"

VALID_CHART_TYPES = [
    "bar charts",
    "line charts",
    "pie charts",
    "mixed graphs",
    "radar charts"
]

# 🔴 UPDATE YOUR MYSQL CREDENTIALS HERE
MYSQL_USER = "root"
MYSQL_PASSWORD = "nandhu"
MYSQL_HOST = "localhost"
MYSQL_DB = "indiabix"


# ==============================
# DATABASE SETUP
# ==============================

Base = declarative_base()

class Question(Base):
    __tablename__ = "datainterpretation"

    id = Column(Integer, primary_key=True, autoincrement=True)
    section = Column(Text)
    topic_name = Column(Text)
    sub_section = Column(Text)
    page_number = Column(Integer)
    image_url = Column(Text)
    question_number = Column(Integer)
    question = Column(Text)
    option_a = Column(Text)
    option_b = Column(Text)
    option_c = Column(Text)
    option_d = Column(Text)
    correct_option = Column(Text)
    correct_answer = Column(Text)
    explanation = Column(Text)


engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}/{MYSQL_DB}"
)

Base.metadata.create_all(engine)

Session = sessionmaker(bind=engine)
session = Session()


# ==============================
# DELAYS
# ==============================

def human_delay():
    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))


def small_delay():
    time.sleep(random.uniform(1, 2))


# ==============================
# DRIVER
# ==============================

def create_driver():
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")

    driver = webdriver.Chrome(options=options)

    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )

    return driver


# ==============================
# HELPERS
# ==============================

def get_chart_image(driver):
    try:
        images = driver.find_elements(By.TAG_NAME, "img")
        for img in images:
            src = img.get_attribute("src")
            if src and any(k in src.lower() for k in ["chart", "graph", "diagram"]):
                return src
    except:
        pass
    return ""


def get_explanation(block):
    try:
        exp = block.find_elements(By.CLASS_NAME, "bix-ans-description")
        if exp:
            return exp[0].text.strip()
    except:
        pass
    return ""


def go_to_next_page(driver):
    try:
        next_li = driver.find_element(
            By.XPATH,
            "//ul[contains(@class,'pagination')]//li[.//a[normalize-space()='Next']]"
        )

        if "disabled" in next_li.get_attribute("class").lower():
            return False

        next_btn = next_li.find_element(By.TAG_NAME, "a")
        driver.execute_script("arguments[0].click();", next_btn)

        human_delay()
        return True

    except:
        return False


# ==============================
# SAVE TO MYSQL
# ==============================

def save_to_db(data):
    try:
        row = Question(**data)
        session.add(row)
        session.commit()
    except Exception as e:
        session.rollback()
        print("DB Error:", e)


# ==============================
# SCRAPER
# ==============================

def scrape_exercise(driver, chart_type, topic_name):

    wait = WebDriverWait(driver, 10)
    page_number = 1

    while True:

        human_delay()
        chart_image = get_chart_image(driver)

        blocks = wait.until(
            lambda d: d.find_elements(By.CLASS_NAME, "bix-div-container")
        )

        print(f"{chart_type} | Page {page_number} → {len(blocks)}")

        for idx, block in enumerate(blocks, start=1):

            try:
                question = block.find_element(By.CLASS_NAME, "bix-td-qtxt").text.strip()
            except:
                continue

            option_rows = block.find_elements(By.CLASS_NAME, "bix-opt-row")

            option_texts = []
            correct_answer = ""
            correct_option = ""
            labels = ["A", "B", "C", "D"]

            for row in option_rows:
                try:
                    val = row.find_element(By.CLASS_NAME, "bix-td-option-val")
                    option_texts.append(val.text.strip())
                except:
                    option_texts.append("")

            while len(option_texts) < 4:
                option_texts.append("")

            # click options
            for row in option_rows:
                try:
                    btn = row.find_element(By.CLASS_NAME, "bix-td-option")
                    driver.execute_script("arguments[0].click();", btn)
                    small_delay()
                except:
                    pass

            time.sleep(1)

            # detect correct
            for i, row in enumerate(option_rows):
                try:
                    val = row.find_element(By.CLASS_NAME, "bix-td-option-val")
                    cls = val.get_attribute("class") or ""
                    if "correct" in cls.lower() or "right" in cls.lower():
                        correct_answer = val.text.strip()
                        correct_option = labels[i]
                        break
                except:
                    pass

            explanation = get_explanation(block)

            # 🔴 SAVE TO DB
            save_to_db({
                "section": SECTION,
                "topic_name": chart_type,
                "sub_section": topic_name,
                "page_number": page_number,
                "image_url": chart_image,
                "question_number": idx,
                "question": question,
                "option_a": option_texts[0],
                "option_b": option_texts[1],
                "option_c": option_texts[2],
                "option_d": option_texts[3],
                "correct_option": correct_option,
                "correct_answer": correct_answer,
                "explanation": explanation
            })

        if not go_to_next_page(driver):
            break

        page_number += 1


# ==============================
# MAIN
# ==============================

def main():

    driver = create_driver()
    wait = WebDriverWait(driver, 15)

    driver.get(BASE_URL)
    human_delay()

    links = wait.until(
        lambda d: d.find_elements(By.XPATH, "//ul[@class='need-ul-filter']//a")
    )

    chart_types = {}

    for link in links:
        name = link.text.strip().lower()
        url = link.get_attribute("href")

        if any(t in name for t in VALID_CHART_TYPES):
            chart_types[name.title()] = url

    for chart_type, chart_url in chart_types.items():

        print("\nProcessing:", chart_type)

        driver.get(chart_url)
        human_delay()

        exercise_links = driver.find_elements(
            By.XPATH,
            "//a[contains(@href,'/data-interpretation/') and string-length(normalize-space(text())) > 0]"
        )

        exercises = []

        for e in exercise_links:
            name = e.text.strip()
            url = e.get_attribute("href")

            nums = re.findall(r"\d+", name)
            if nums:
                exercises.append((int(nums[0]), url, name))

        exercises.sort(key=lambda x: x[0])

        for _, ex_url, topic_name in exercises:
            driver.get(ex_url)
            human_delay()
            scrape_exercise(driver, chart_type, topic_name)

    driver.quit()
    print("✅ DONE")


if __name__ == "__main__":
    main()


Processing: Bar Charts
Bar Charts | Page 1 → 5
Bar Charts | Page 1 → 5
Bar Charts | Page 1 → 5
Bar Charts | Page 2 → 1
Bar Charts | Page 1 → 5
Bar Charts | Page 1 → 5
Bar Charts | Page 1 → 5
Bar Charts | Page 1 → 5
Bar Charts | Page 1 → 5
Bar Charts | Page 1 → 5
Bar Charts | Page 1 → 5
Bar Charts | Page 1 → 4
Bar Charts | Page 1 → 5
Bar Charts | Page 1 → 4
Bar Charts | Page 1 → 4
Bar Charts | Page 1 → 5
Bar Charts | Page 1 → 3
Bar Charts | Page 1 → 5
Bar Charts | Page 1 → 5
Bar Charts | Page 1 → 5

Processing: Pie Charts
Pie Charts | Page 1 → 5
Pie Charts | Page 2 → 4
Pie Charts | Page 1 → 3
Pie Charts | Page 1 → 4
Pie Charts | Page 1 → 3
Pie Charts | Page 1 → 3
Pie Charts | Page 1 → 5
Pie Charts | Page 1 → 5
Pie Charts | Page 1 → 4
Pie Charts | Page 1 → 5
Pie Charts | Page 2 → 1
Pie Charts | Page 1 → 5
Pie Charts | Page 1 → 5
Pie Charts | Page 1 → 5
Pie Charts | Page 1 → 5

Processing: Line Charts
Line Charts | Page 1 → 5
Line Charts | Page 1 → 5
Line Charts | Page 1 → 5
Line Charts 